In [77]:
#import 

import numpy as np 
import sympy as sp 
import matplotlib.pyplot as plt


In [78]:
# Define our state 
x, y, z, x_dot, y_dot, z_dot, theta, phi, gamma, theta_dot, phi_dot, gamma_dot, n_1, n_2, n_3, n_4 = sp.symbols(r'x y z \dot{x} \dot{y} \dot{z} theta phi gamma \dot{\theta} \dot{\phi} \dot{\gamma} n_1 n_2 n_3 n_4')
state = sp.Matrix([x, y, z, x_dot, y_dot, z_dot, theta, phi, gamma, theta_dot, phi_dot, gamma_dot])


u = sp.Matrix([n_1, n_2, n_3, n_4])

# Define constants 

Ixx, Iyy, Izz = 1,1,1 # place holders for now
Inertia = sp.diag(Ixx, Iyy,Izz)

m = 0.1 # kg place holder for now 
g = 9.81 

l = 0.16 # distance from the COM to the motor  PLACEHOLDER
angle_arm = sp.pi/3 # angle relative to the arm and the horizontal PLACEHOLDER

# Define the Rotation Matrix

def get_R_ItoB(theta, phi, gamma): 
    R_ItoB = sp.Matrix([
        [sp.cos(gamma)*sp.cos(phi), sp.sin(gamma)*sp.cos(phi), -sp.sin(phi)],
        [sp.cos(gamma)*sp.sin(phi)*sp.sin(theta) - sp.sin(gamma)*sp.cos(theta), sp.sin(gamma)*sp.sin(phi)*sp.sin(theta) + sp.cos(gamma)*sp.cos(theta), sp.cos(phi)*sp.sin(theta)],
        [sp.cos(gamma)*sp.sin(phi)*sp.cos(theta) + sp.sin(gamma)*sp.sin(theta), sp.sin(gamma)*sp.sin(phi)*sp.cos(theta) - sp.cos(gamma)*sp.sin(theta), sp.cos(phi)*sp.cos(theta)]
    ])
    return R_ItoB

# u


In [79]:
# Define the RPM to thurst equation 

# rpm = sp.Matrix([1,1,1,1]) # 

p = 1.225 # kg/m^3 air density
CT = 0.9 # Thrust coefficient from propeller
D = 0.0254 # Propeller diameter 

def get_thrust(rpm_vec):
    thrust = sp.Matrix([0, 0, (CT * p * D**4 * rpm_vec.applyfunc(lambda x: (x/60)**2)).dot(sp.ones(4,1))]) 
    return thrust # thrust from rpm

def get_thrust_components(rpm_vec):
    return (CT * p * D**4 * rpm_vec.applyfunc(lambda x: (x/60)**2))
    
#get_thrust()
# sp.ones(4,1)


# Define the torque equations and lever arms 
angles_arms = sp.Matrix([angle_arm,sp.pi - angle_arm, angle_arm + sp.pi, -angle_arm])

def get_torques(u):
    T_vec = get_thrust_components(u)
    tau = sp.zeros(3, 1)
    for T, angle in zip(T_vec, angles_arms):
        r = l * sp.Matrix([sp.cos(angle), sp.sin(angle), 0])
        tau += r.cross(sp.Matrix([0, 0, T]))
    return tau


        

In [80]:
# position ODE 

def ODE_position(state, u):
    
    v_body = sp.Matrix(state[3:6])
    
    angular_v = sp.Matrix(state[9:12]) # angular body 
    F_thrust = get_thrust(u)

    R_IB = get_R_ItoB(*state[6:9])
    F_g = R_IB * sp.Matrix([0,0,-m*g])
    F_body = F_thrust + F_g
    v_body_dot = F_body/m - angular_v.cross(v_body)
    return v_body_dot

def ODE_kinematics(state):
    R_BI = get_R_ItoB(*state[6:9]).T # same as inverse = transpose for rotation matrix
    return R_BI * state[3:6]

# ODE_position(sp.zeros(12,1), sp.zeros(4,1))

In [81]:
# rotational ODE

def ODE_rotational(state,u):
    angular_vel = sp.Matrix(state[9:])
    tau = get_torques(angle_arm_vec=angles_arms, u=u)
    angular_accel_body = Inertia.inv() * (tau - angular_vel.cross(Inertia*angular_vel) )
    return angular_accel_body

